In [ ]:
!pip install roboflow ultralytics -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/MasterFlowers"
os.makedirs(BASE_FOLDER, exist_ok=True)

print(f"Folder creado: {BASE_FOLDER}")

In [ ]:
import torch

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

# Key lives in Colab Secrets (key icon in the sidebar), never in the notebook.
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("myworkspace-rde28").project("flowers_segmentation-lpvdz")
version = project.version(1)
dataset = version.download("yolov11")


## This is the dataset I sue to train this YOLOv11-small-seg model

flowers_segmentation Computer Vision Dataset
https://universe.roboflow.com/flowersdetection/flowers_segmentation

In [ ]:
!ls  { dataset.location}

In [ ]:
print(f"Dataset is at : {dataset.location}")

In [ ]:
import os
import shutil
import yaml
from collections import Counter
from pathlib import Path

## Class filtering and re-indexing

The Roboflow dataset is long-tailed: many classes carry too few labelled instances to learn from. Every class with fewer than `MIN_INSTANCES = 48` instances, counted across train/valid/test, is dropped and the survivors are renumbered contiguously from 0 by `old_to_new_id`.

**Model class IDs are therefore not Roboflow class IDs.** The authoritative list is `new_class_names`, written into the filtered `data.yaml` and later embedded in the ONNX `names` metadata. Downstream code must read the names from the model, never hardcode an index.

Images that lose their only visible instance are dropped along with their labels.

In [ ]:
DATASET_ROOT = dataset.location
OUTPUT_ROOT = os.path.join(os.path.dirname(DATASET_ROOT), "flowers_seg_filtered")
MIN_INSTANCES = 48

In [ ]:
with open(os.path.join(DATASET_ROOT, "data.yaml")) as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml["names"]

In [ ]:

instance_counts = Counter()
for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(DATASET_ROOT, split, "labels")
    if not os.path.isdir(labels_dir):
        continue
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    instance_counts[int(parts[0])] += 1


In [ ]:
kept_old_ids = sorted(cid for cid, count in instance_counts.items() if count >= MIN_INSTANCES)
dropped_old_ids = set(range(len(class_names))) - set(kept_old_ids)

In [ ]:
old_to_new_id = {old_id: new_id for new_id, old_id in enumerate(kept_old_ids)}
new_class_names = [class_names[old_id] for old_id in kept_old_ids]


In [ ]:
print(f"Clases originales: {len(class_names)}")
print(f"Clases conservadas: {len(new_class_names)}")
print(f"Clases descartadas: {len(dropped_old_ids)}")
print()

In [ ]:
images_kept_total = 0
images_dropped_total = 0

In [ ]:
for split in ["train", "valid", "test"]:
    src_images_dir = os.path.join(DATASET_ROOT, split, "images")
    src_labels_dir = os.path.join(DATASET_ROOT, split, "labels")
    if not os.path.isdir(src_labels_dir):
        continue

    dst_images_dir = os.path.join(OUTPUT_ROOT, split, "images")
    dst_labels_dir = os.path.join(OUTPUT_ROOT, split, "labels")
    os.makedirs(dst_images_dir, exist_ok=True)
    os.makedirs(dst_labels_dir, exist_ok=True)

    split_kept = 0
    split_dropped = 0

    for label_file in os.listdir(src_labels_dir):
        if not label_file.endswith(".txt"):
            continue

        src_label_path = os.path.join(src_labels_dir, label_file)
        with open(src_label_path) as f:
            original_lines = [line.strip().split() for line in f if line.strip()]

        had_annotations = len(original_lines) > 0

        remapped_lines = []
        for parts in original_lines:
            old_id = int(parts[0])
            if old_id in old_to_new_id:
                new_id = old_to_new_id[old_id]
                remapped_lines.append(" ".join([str(new_id)] + parts[1:]))

        if had_annotations and len(remapped_lines) == 0:
            split_dropped += 1
            continue

        stem = Path(label_file).stem
        image_file = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = os.path.join(src_images_dir, stem + ext)
            if os.path.exists(candidate):
                image_file = candidate
                break

        if image_file is None:
            continue

        shutil.copy2(image_file, os.path.join(dst_images_dir, os.path.basename(image_file)))
        with open(os.path.join(dst_labels_dir, label_file), "w") as f:
            f.write("\n".join(remapped_lines) + ("\n" if remapped_lines else ""))

        split_kept += 1

    print(f"{split}: {split_kept} imagenes conservadas, {split_dropped} descartadas por perder su unica instancia")
    images_kept_total += split_kept
    images_dropped_total += split_dropped


In [ ]:
new_data_yaml = {
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(new_class_names),
    "names": new_class_names,
}

In [ ]:
with open(os.path.join(OUTPUT_ROOT, "data.yaml"), "w") as f:
    yaml.dump(new_data_yaml, f, default_flow_style=False, sort_keys=False)

In [ ]:
print(f"\nTotal imagenes conservadas: {images_kept_total}")
print(f"Total imagenes descartadas (perdieron su unica instancia visible): {images_dropped_total}")
print(f"\nDataset filtrado guardado en: {OUTPUT_ROOT}")

## Test split, rarest class first

Roboflow's own split leaves some classes with no test representation. Here 7.5% of train is moved to test, walking classes from rarest to most common and taking at least one image each, so evaluation covers the long tail rather than only the common flowers. `random.seed(42)` keeps the split reproducible.

Images are **moved, not copied**: re-running these cells re-splits an already reduced train set.

In [ ]:
import math
import random
from pathlib import Path
from collections import defaultdict

In [ ]:
random.seed(42)

In [ ]:
FILTERED_ROOT = "/content/flowers_seg_filtered"
TEST_FRACTION = 0.075



In [ ]:
train_images_dir = os.path.join(FILTERED_ROOT, "train", "images")
train_labels_dir = os.path.join(FILTERED_ROOT, "train", "labels")
test_images_dir = os.path.join(FILTERED_ROOT, "test", "images")
test_labels_dir = os.path.join(FILTERED_ROOT, "test", "labels")

In [ ]:
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_labels_dir, exist_ok=True)


In [ ]:
label_files = [f for f in os.listdir(train_labels_dir) if f.endswith(".txt")]

In [ ]:
class_to_images = defaultdict(set)
image_classes = {}

In [ ]:
for label_file in label_files:
    stem = Path(label_file).stem
    with open(os.path.join(train_labels_dir, label_file)) as f:
        class_ids = set()
        for line in f:
            parts = line.strip().split()
            if parts:
                class_ids.add(int(parts[0]))
    image_classes[stem] = class_ids
    for cid in class_ids:
        class_to_images[cid].add(stem)

In [ ]:
class_order = sorted(class_to_images.keys(), key=lambda c: len(class_to_images[c]))


In [ ]:
assigned_to_test = set()

In [ ]:
for cid in class_order:
    available = list(class_to_images[cid] - assigned_to_test)
    if not available:
        continue

    n_target = max(1, math.ceil(len(class_to_images[cid]) * TEST_FRACTION))
    n_take = min(n_target, len(available))
    random.shuffle(available)
    chosen = available[:n_take]
    assigned_to_test.update(chosen)

In [ ]:
print(f"Total imagenes asignadas a test: {len(assigned_to_test)}")

In [ ]:
def find_image_file(stem, images_dir):
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = os.path.join(images_dir, stem + ext)
        if os.path.exists(candidate):
            return candidate
    return None

In [ ]:

moved = 0

In [ ]:

for stem in assigned_to_test:
    img_path = find_image_file(stem, train_images_dir)
    label_path = os.path.join(train_labels_dir, stem + ".txt")
    if img_path is None or not os.path.exists(label_path):
        continue
    shutil.move(img_path, os.path.join(test_images_dir, os.path.basename(img_path)))
    shutil.move(label_path, os.path.join(test_labels_dir, os.path.basename(label_path)))
    moved += 1

In [ ]:
print(f"Imagenes movidas fisicamente a test: {moved}")
print(f"Imagenes restantes en train: {len(os.listdir(train_images_dir))}")

In [ ]:
test_class_coverage = defaultdict(int)
for stem in assigned_to_test:
    for cid in image_classes.get(stem, []):
        test_class_coverage[cid] += 1

In [ ]:

missing_classes = [cid for cid in class_to_images if cid not in test_class_coverage]
print(f"\nClases sin ninguna instancia en test: {len(missing_classes)}")
if missing_classes:
    print(missing_classes)

## Training

YOLOv11s-seg fine-tuned at 640px, AdamW, 150 epochs with `patience=20` for early stopping. Augmentation stays moderate on purpose - `degrees=10` and `flipud=0.1` are low because flower photographs are almost always upright.

The model is trained with segmentation masks, but the Java backend uses only boxes and class scores.

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11s-seg.pt")

In [ ]:
results = model.train(
    data="/content/flowers_seg_filtered/data.yaml",
    epochs=150,
    imgsz=640,
    batch=32,
    patience=20,
    device=0,
    optimizer="AdamW",
    lr0=1.5e-3,
    augment=True,
    degrees=10,
    fliplr=0.5,
    flipud=0.1,
    mosaic=0.8,
    mixup=0.1,
    project=BASE_FOLDER,
    name="flowers_segv",
)

In [ ]:
results_val = model.val(
    data=os.path.join(FILTERED_ROOT, "data.yaml"),
    imgsz=640,
    batch=32,
    device=0,
    project=BASE_FOLDER,
    name="flowers_segv_val",
)

In [ ]:
results_test = model.val(
    data=os.path.join(FILTERED_ROOT, "data.yaml"),
    imgsz=640,
    batch=32,
    device=0,
    split="test", # Specify the test set
    project=BASE_FOLDER,
    name="flowers_segv_test", # Update the name for test results
)

## Results

Final metrics for the 90 surviving classes, recorded here because notebook outputs are stripped before commit.

| split | images | instances | Box mAP50 | Box mAP50-95 | Box P | Box R |
|---|---|---|---|---|---|---|
| valid | 991 | 1447 | **0.920** | 0.844 | 0.863 | 0.869 |
| test | 472 | 697 | **0.925** | 0.854 | 0.914 | 0.863 |

Mask metrics track the box metrics closely (test mask mAP50 `0.929`), though the Java backend uses boxes only.

Inference is `6.5 ms` per image at 640px on the Colab GPU.

Test scores match validation rather than falling below it, which is the signal that the 150 epochs with `patience=20` stopped before overfitting. The held-out test split was built rarest-class-first, so this is not an easy subset.

`cm_matrix.png` in the repo root is the 90-class confusion matrix for the test split.

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/MasterFlowers/flowers_segv/weights/best.pt"

In [ ]:
model = YOLO(MODEL_PATH)

## ONNX export and the output contract

`opset=17`, `simplify=True`, `dynamic=False` - a fixed `[1, 3, 640, 640]` input, which is what the ONNX Runtime adapter in the Java backend expects.

The verification cells below print the contract that adapter depends on:

- `output0` `[1, 126, 8400]` - **channels first**: 4 box + 90 class scores + 32 mask coefficients. Element `(c, a)` sits at `c*8400 + a`.
- `output1` `[1, 32, 160, 160]` - mask prototypes, unused downstream.

Export defaults to `nms=False`, so non-maximum suppression is implemented in Java.

In [ ]:

model.export(
    format="onnx",
    opset=17,
    simplify=True,
    imgsz=640,
    dynamic=False,
    half=False,
)

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

In [ ]:
ONNX_PATH = MODEL_PATH.replace(".pt", ".onnx")

In [ ]:
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("Modelo ONNX valido")

In [ ]:
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])


In [ ]:
print("\nInputs:")
for inp in session.get_inputs():
    print(f"  {inp.name}: {inp.shape} ({inp.type})")


In [ ]:
print("\nOutputs:")
for out in session.get_outputs():
    print(f"  {out.name}: {out.shape} ({out.type})")


In [ ]:
dummy_input = np.random.rand(1, 3, 640, 640).astype(np.float32)
outputs = session.run(None, {session.get_inputs()[0].name: dummy_input})

In [ ]:
for i, out in enumerate(outputs):
    print(f"Output {i} shape: {out.shape}")